In [18]:
import pandas as pd
import numpy as np
import re
import h5py
from utils.template import sample_space, affine
from utils.kernel import kernel_calc

In [3]:
df = pd.read_csv("simon_func_mni_2019_Feb_19.txt", sep="\t", header=None)
df = df[(df.loc[:,13].str.contains("Normal Mapping")) & (df.loc[:,20] > 8) & (df.loc[:,20] < 50)].reset_index(drop=True)

lines_columns = ['Author', 'Subjects', 'Cond', 'ExpIndex']
lines = pd.DataFrame(columns=lines_columns)

lines["Author"] = df.loc[:,2]
lines["Subjects"] = df.loc[:,20]
lines["Cond"] = df.loc[:,6:17].to_string(header=False, index=False)

cnt_exp = 0
first_lines = [0]
for i in range(lines.shape[0]):
    if i > 0:
        cnt_exp += 1
        if (lines.loc[i, ['Author', 'Subjects', 'Cond']] == lines.loc[i-1, ['Author', 'Subjects', 'Cond']]).all():
            cnt_exp -= 1
        else:
            first_lines.append(i)
    lines.at[i, 'ExpIndex'] = cnt_exp

num_exp = cnt_exp + 1

exp_info = lines.loc[first_lines]
exp_info = exp_info.drop("Cond", axis=1).drop("ExpIndex", axis=1)
exp_info["Foci"] = np.unique(lines["ExpIndex"], return_counts=True)[1]
exp_info = exp_info[exp_info["Foci"] < 20].reset_index(drop=True)

sigma = open("SD", "r").read().splitlines()
sigma = [float(number[:-4].strip()) for number in sigma]

true_coordinate = [20,20,20]
with h5py.File("simulation_data.hdf5", "w") as f:
    for num_studies in range(15,46):
        for true_activations in range(10):
            for rep in range(100):
                sample_sizes = []
                random_foci = []
                for i in range(num_studies):
                    study_id = np.random.randint(exp_info.shape[0])
                    sample_sizes.append(exp_info.loc[study_id, "Subjects"])
                    num_foci = exp_info.loc[study_id, "Foci"]
                    random_foci.append(sample_space[:,np.random.randint(0,sample_space.shape[1], num_foci)].T)
                for j in range(true_activations):
                    random_foci[j][0] = true_coordinate + (np.random.random((1,3)) * np.random.choice([-1,1], 3) * np.random.choice(sigma) * 0.5)
                f.create_dataset(f"{num_studies}/{true_activations}/{rep}/sample_sizes", data=np.asarray(sample_sizes))
                for idx, arr in enumerate(random_foci):   
                    f.create_dataset(f"{num_studies}/{true_activations}/{rep}/foci/{idx}", data=arr)
